<a href="https://colab.research.google.com/github/Sangeetha3315/Agentic-AI-and-computer-vision-workshop-projects/blob/main/RAG_research_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers faiss-cpu pypdf groq arxiv requests
print("Installed.")

In [ ]:
import os
import re
import textwrap
import numpy as np
from getpass import getpass

import faiss                                    # fast vector search
from sentence_transformers import SentenceTransformer   # turns text into vectors
import arxiv                                     # download real papers
import requests
from pypdf import PdfReader                      # read PDF text
from groq import Groq                            # talk to the LLM

print("Imports ready.")

In [ ]:
# Paste a free Groq API key here (get one at https://console.groq.com/keys — no credit card needed).
# The input box below hides what you type.
os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ")

client = Groq()
MODEL = "openai/gpt-oss-120b"

def ask_llm(prompt: str, system: str = "", max_tokens: int = 600) -> str:
    """
    Sends one message to the LLM and returns its reply as plain text.
    Every other cell in this notebook calls this ONE function whenever it needs the LLM to think —
    that's the only place the model is actually called.
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=max_tokens,
        temperature=0,          # 0 = give the most likely answer, not a random/creative one
        messages=messages,
    )
    return response.choices[0].message.content

# quick test
print(ask_llm("Reply with exactly: LLM connection OK"))

In [ ]:
# The exact papers we want, by their official arXiv ID.
PAPERS = [
    {"id": "1706.03762", "label": "Attention Is All You Need (the original Transformer paper)"},
    {"id": "2005.11401", "label": "Retrieval-Augmented Generation (the original RAG paper)"},
    {"id": "2210.03629", "label": "ReAct (reasoning + acting in AI agents)"},
]

os.makedirs("papers", exist_ok=True)
paper_files = []   # will hold {id, title, path} for each downloaded paper

client_arxiv = arxiv.Client()
for p in PAPERS:
    search = arxiv.Search(id_list=[p["id"]])
    result = next(client_arxiv.results(search))

    filepath = f"papers/{result.get_short_id()}.pdf"
    pdf_bytes = requests.get(result.pdf_url, timeout=30).content
    with open(filepath, "wb") as f:
        f.write(pdf_bytes)

    paper_files.append({"id": result.get_short_id(), "title": result.title, "path": filepath})
    print(f"Downloaded: {result.title}")

print(f"\n {len(paper_files)} papers saved to ./papers/")

In [ ]:
def get_pdf_text(path: str) -> str:
    """Reads every page of a PDF and joins the text together."""
    reader = PdfReader(path)
    pages = [page.extract_text() or "" for page in reader.pages]
    text = "\n".join(pages)
    text = re.sub(r"\s+", " ", text)   # collapse messy whitespace into single spaces
    return text

def split_into_chunks(text: str, chunk_size: int = 800, overlap: int = 100):
    """
    Cuts text into overlapping chunks of `chunk_size` characters.
    `overlap` characters are repeated at the start of the next chunk, so an idea that
    spans a chunk boundary doesn't get lost.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# Build our full list of chunks, across all 3 papers
all_chunks = []   # each item: {"id": ..., "title": ..., "text": ...}

for paper in paper_files:
    full_text = get_pdf_text(paper["path"])
    text_chunks = split_into_chunks(full_text)
    for i, chunk_text in enumerate(text_chunks):
        all_chunks.append({
            "id": f"{paper['id']}_chunk{i}",
            "title": paper["title"],
            "text": chunk_text,
        })

print(f" Created {len(all_chunks)} chunks from {len(paper_files)} papers")
print("\nExample chunk:")
print(textwrap.fill(all_chunks[10]["text"][:400], 100))

In [ ]:
print("Loading the embedding model (turns text into vectors)...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Turn every chunk's text into a vector
chunk_texts = [c["text"] for c in all_chunks]
vectors = embed_model.encode(chunk_texts, show_progress_bar=True, normalize_embeddings=True)
vectors = np.array(vectors, dtype="float32")

print(f"\nEach chunk became a vector with {vectors.shape[1]} numbers.")
print(f"Example — first 8 numbers of chunk 0's vector: {vectors[0][:8]}")

# Store all vectors in a FAISS index for fast searching
index = faiss.IndexFlatIP(vectors.shape[1])   # IP = inner product; with normalized vectors this IS cosine similarity
index.add(vectors)

print(f"\n {index.ntotal} vectors stored and ready to search.")

In [ ]:
def find_relevant_chunks(question: str, top_k: int = 4):
    """Returns the top_k chunks whose meaning is closest to the question."""
    question_vector = embed_model.encode([question], normalize_embeddings=True)
    similarities, indices = index.search(np.array(question_vector, dtype="float32"), top_k)

    results = []
    for idx, score in zip(indices[0], similarities[0]):
        results.append({"chunk": all_chunks[idx], "similarity": float(score)})
    return results

# Try it
test_question = "What is positional encoding used for?"
matches = find_relevant_chunks(test_question)

print(f"Question: {test_question}\n")
for m in matches:
    print(f"[similarity = {m['similarity']:.3f}]  {m['chunk']['id']}  ({m['chunk']['title'][:40]}...)")
    print(textwrap.fill(m["chunk"]["text"][:220], 100))
    print()

In [ ]:
ANSWER_INSTRUCTIONS = (
    "You are a helpful research assistant. Answer the question using ONLY the "
    "context provided below. For every fact you state, mention which chunk it came from, like [chunk_id]. "
    "If the context does not contain the answer, say so clearly instead of guessing."
)

def answer_question(question: str, top_k: int = 4):
    matches = find_relevant_chunks(question, top_k=top_k)

    # Build the context block the LLM will read
    context_text = ""
    for m in matches:
        c = m["chunk"]
        context_text += f"\n[{c['id']}] (from: {c['title']})\n{c['text']}\n"

    prompt = f"Context:\n{context_text}\n\nQuestion: {question}\n\nAnswer:"
    answer = ask_llm(prompt, system=ANSWER_INSTRUCTIONS, max_tokens=400)
    return answer, matches

# Try the full pipeline end to end
question = "What is positional encoding used for?"
answer, sources = answer_question(question)

print("QUESTION:", question)
print("\nANSWER:")
print(textwrap.fill(answer, 100))
print("\nSOURCES USED:", [m["chunk"]["id"] for m in sources])

In [ ]:
questions_to_try = [
    "Why is attention called 'scaled dot-product attention'?",
    "What is the ReAct pattern for AI agents?",
    "What is the capital of France?",
]

for q in questions_to_try:
    print("="*90)
    print("QUESTION:", q)
    answer, sources = answer_question(q)
    print("\nANSWER:")
    print(textwrap.fill(answer, 100))
    print("\nSOURCES:", [m["chunk"]["id"] for m in sources])
    print()